In [1]:
!pip install -qU datasets pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.


In [2]:
import pandas as pd
from datasets import load_dataset

In [3]:
REPO = "bakrianoo/jabarti-llm-dataset"

pretrain = load_dataset (
    REPO,
    "pretrain",
    split= "phase1_train"
).to_pandas()


finetune = load_dataset (
    REPO,
    "finetune",
    split= "train"
).to_pandas()

README.md:   0%|          | 0.00/7.01k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

pretrain/train-00000-of-00029.parquet: reconstructing file:   0%|          |  0.00B / 75.6MB            

pretrain/train-00000-of-00029.parquet: downloading bytes:           |  0.00B            

pretrain/train-00001-of-00029.parquet: reconstructing file:   0%|          |  0.00B / 75.5MB            

... (74 outputs left out)

finetune/eval-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.10MB            

finetune/eval-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

In [5]:
print("=== pretrain schema ===")
print(pretrain.dtypes)

print("\n=== finetune schema ===")
print(finetune.dtypes)

=== pretrain schema ===
article_id              str
title                   str
url                     str
language                str
category                str
source_type             str
egypt_relevance         str
content_quality         str
topic_domain            str
importance              str
is_person              bool
is_stub_like           bool
ortho_convention        str
pageviews_30d         int64
chunk_strategy          str
chunk_index           int64
chunk_total           int64
pair_id                 str
dataset_version         str
ds_source               str
text                    str
section_index         int64
section_total         int64
ortho_stripped         bool
page_id                 str
record_id               str
sample_weight       float64
dtype: object

=== finetune schema ===
difficulty           str
type                 str
qa_id                str
dataset_version      str
question             str
article_title        str
answer               str
source

In [29]:
splits= {
    "pretrain / phase1_train" : ("pretrain", "phase1_train"),
    "pretrain / phase1_eval" : ("pretrain", "phase1_eval"),
    "pretrain / phase2_train" : ("pretrain", "phase2_train"),
    "pretrain / phase2_eval" : ("pretrain", "phase2_eval"),

    "finetune / train" : ("finetune", "train"),
    "finetune / eval" : ("finetune", "eval"),
}

# Rough token estimate for bilinguial AR/EN with a BPE tokenizer at 24K vocab
CHARS_PER_TOKEN = 3.5

#Candidate model context lengths
CONTEXT_LENGTHS =[512, 1024, 2028]

for label_name, (subset, split_name) in splits.items():

  df = load_dataset (
      REPO , subset, split =split_name
  ).to_pandas()

  print(f"\n==== {label_name} ====")
  print("Rows: ", df.shape[0])

  nulls = df.isnull().sum()
  non_nulls = nulls[ nulls > 0 ]

  print(
      "NONE"
      if non_nulls.empty
      else non_nulls.to_string()
  )



  # ==== Langauge Distribution
  total_rows = df.shape[0]
  lang_counts = df["language"].value_counts()
  for lang, n in lang_counts.items ():
    print(f"{lang} => {n} samples -- ",
          f"{n/total_rows*100:.1f}%"
          )


  # ==== Character Length Distribution
  text_len_dist = 0
  if subset =='pretrain' :
      text_len_dist = df['text'].str.len().describe().round(1)

  elif subset =='finetune' :
      text_len_dist = pd.concat([
          df["question"].str.len(),
          df["answer"].str.len()
      ],ignore_index=True).describe().round(1)

  print("\n=== text_len_dist (Characters) ===")
  print(text_len_dist)

  # ==== Approximate token length per model sample
  if subset == "pretrain" :
      sample_chars = df["text"].str.len()


  elif subset == "finetune" :
      sample_chars = (
        df["question"].str.len()
        +
        df["answer"].str.len()
      )

  approx_tokens = (sample_chars / CHARS_PER_TOKEN).astype(int)
  print("\n=== Approxmate token count per sample === ")
  print(approx_tokens.describe().round(1))

  # === Context length coverage
  for ctx in CONTEXT_LENGTHS:
      dropped_pct = (
           (approx_tokens > ctx).mean()*100
      )
      kept_pct = 100 -dropped_pct

      print(
          f"current_max_length={ctx} | "
          f"kept: {kept_pct:.1f}% |"
          f"dropped: {dropped_pct:.1f}% |"
      )


  print("===== ===== =====")

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]


==== pretrain / phase1_train ====
Rows:  175476
NONE
en => 125766 samples --  71.7%
ar => 49710 samples --  28.3%

=== text_len_dist (Characters) ===
count    175476.0
mean       1553.0
std         702.1
min         178.0
25%        1169.0
50%        1653.0
75%        1913.0
max       20049.0
Name: text, dtype: float64

=== Approxmate token count per sample === 
count    175476.0
mean        443.3
std         200.6
min          50.0
25%         334.0
50%         472.0
75%         546.0
max        5728.0
Name: text, dtype: float64
current_max_length=512 | kept: 61.9% |dropped: 38.1% |
current_max_length=1024 | kept: 99.3% |dropped: 0.7% |
current_max_length=2028 | kept: 99.8% |dropped: 0.2% |
===== ===== =====


Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]


==== pretrain / phase1_eval ====
Rows:  2000
NONE
en => 1426 samples --  71.3%
ar => 574 samples --  28.7%

=== text_len_dist (Characters) ===
count    2000.0
mean     1565.1
std       577.7
min       268.0
25%      1226.8
50%      1666.5
75%      1928.0
max      7520.0
Name: text, dtype: float64

=== Approxmate token count per sample === 
count    2000.0
mean      446.7
std       165.1
min        76.0
25%       350.0
50%       476.0
75%       550.0
max      2148.0
Name: text, dtype: float64
current_max_length=512 | kept: 61.4% |dropped: 38.6% |
current_max_length=1024 | kept: 99.4% |dropped: 0.6% |
current_max_length=2028 | kept: 99.9% |dropped: 0.1% |
===== ===== =====


... (6 outputs left out)

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]


==== finetune / train ====
Rows:  57488
NONE
ar => 44947 samples --  78.2%
en => 12541 samples --  21.8%

=== text_len_dist (Characters) ===
count    114976.0
mean         92.1
std          53.8
min          10.0
25%          56.0
50%          77.0
75%         111.0
max         423.0
dtype: float64

=== Approxmate token count per sample === 
count    57488.0
mean        52.2
std         21.9
min          9.0
25%         35.0
50%         48.0
75%         69.0
max        197.0
dtype: float64
current_max_length=512 | kept: 100.0% |dropped: 0.0% |
current_max_length=1024 | kept: 100.0% |dropped: 0.0% |
current_max_length=2028 | kept: 100.0% |dropped: 0.0% |
===== ===== =====


Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]


==== finetune / eval ====
Rows:  6379
NONE
ar => 4976 samples --  78.0%
en => 1403 samples --  22.0%

=== text_len_dist (Characters) ===
count    12758.0
mean        92.9
std         54.6
min         11.0
25%         56.0
50%         78.0
75%        113.0
max        515.0
dtype: float64

=== Approxmate token count per sample === 
count    6379.0
mean       52.7
std        21.9
min        10.0
25%        35.0
50%        49.0
75%        70.0
max       195.0
dtype: float64
current_max_length=512 | kept: 100.0% |dropped: 0.0% |
current_max_length=1024 | kept: 100.0% |dropped: 0.0% |
current_max_length=2028 | kept: 100.0% |dropped: 0.0% |
===== ===== =====


In [17]:
df = load_dataset (
      REPO , "pretrain", split ="phase1_train"
  ).to_pandas()

text_dups = df.duplicated(
    subset=["text"],

).sum()

print(f"Rows with exact duplicate text: {text_dups}")


pair_dups = df.duplicated(
    subset = ["article_id", "chunk_index"]

).sum()

print(f"Duplicate (article_id, chunck_index) pairs: {pair_dups}")


Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Rows with exact duplicate text: 0
Duplicate (article_id, chunck_index) pairs: 0
